# Week 5 — Grad-CAM Explainability + Faithfulness Test

**Project:** Can We Trust AI-Image Detectors on Ugandan Content?
**Member:** AW
**Goal:** Generate Grad-CAM heat maps on test samples, then run a faithfulness test:
hide the highlighted area vs hide a random area — a faithful heat map causes a bigger confidence drop.

### RAM-safe design
- 128×128 images, batch 8
- Grad-CAM on 8 test samples
- Run cells one at a time

### Drive paths
- Root: `/content/drive/MyDrive/Deepfake Detection Project/datasets`
- Outputs: `reports/week5_gradcam_visualization.png`, `reports/week5_gradcam_results.json`

In [ ]:
!pip -q install torch torchvision scikit-learn pandas pillow

In [ ]:
import os, sys, json, glob, gc
from datetime import datetime
import numpy as np
import pandas as pd
from PIL import Image as PILImage
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision import transforms
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
print('PyTorch:', torch.__version__)
print('Imports OK')

## 1. Mount Drive + Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
ROOT = '/content/drive/MyDrive/Deepfake Detection Project/datasets'
MEMBER = 'AW'
IMG_SIZE = 128
BATCH_SIZE = 8
SEED = 42
N_SAMPLES = 8

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

for d in ['reports', 'metadata', 'audit_outputs']:
    os.makedirs(os.path.join(ROOT, d), exist_ok=True)

print('Drive root:', ROOT)

## 2. Load test images (same split as Week 4)

In [ ]:
real_dir = os.path.join(ROOT, 'pilot_data', 'real')
ai_dir = os.path.join(ROOT, 'pilot_data', 'ai')

def load_public_images(real_dir, ai_dir, n_each, img_size):
    paths, labels, meta = [], [], []
    real_files = sorted(glob.glob(os.path.join(real_dir, '*.jpg'))) + sorted(glob.glob(os.path.join(real_dir, '*.png')))
    for p in real_files[:n_each]:
        paths.append(p); labels.append(0)
        meta.append({'path': p, 'label': 'real', 'source': 'pilot'})
    ai_files = sorted(glob.glob(os.path.join(ai_dir, '*.jpg'))) + sorted(glob.glob(os.path.join(ai_dir, '*.png')))
    for p in ai_files[:n_each]:
        paths.append(p); labels.append(1)
        meta.append({'path': p, 'label': 'ai', 'source': 'pilot'})
    if len(paths) == 0:
        print('No pilot images — creating synthetic set.')
        for i in range(n_each):
            arr = np.random.randint(0, 255, (img_size, img_size, 3), dtype=np.uint8)
            im = PILImage.fromarray(arr)
            p = os.path.join(real_dir, f'synthetic_real_{i:03d}.jpg')
            im.save(p, quality=95)
            paths.append(p); labels.append(0)
            meta.append({'path': p, 'label': 'real', 'source': 'synthetic'})
            arr = np.zeros((img_size, img_size, 3), dtype=np.uint8)
            for x in range(img_size):
                arr[:, x, :] = [int(255 * x / img_size), 128, 64]
            im = PILImage.fromarray(arr)
            p = os.path.join(ai_dir, f'synthetic_ai_{i:03d}.jpg')
            im.save(p, quality=95)
            paths.append(p); labels.append(1)
            meta.append({'path': p, 'label': 'ai', 'source': 'synthetic'})
    return paths, labels, meta

n_each = 100
public_paths, public_labels, public_meta = load_public_images(real_dir, ai_dir, n_each, IMG_SIZE)

kampala_dirs = {
    'orig': os.path.join(ROOT, 'kampala', 'originals'),
    'wa_photo': os.path.join(ROOT, 'kampala', 'whatsapp_photo'),
    'wa_doc': os.path.join(ROOT, 'kampala', 'whatsapp_document'),
}
kampala_log_path = os.path.join(ROOT, 'metadata', 'kampala_log.csv')
kampala_paths, kampala_labels, kampala_meta = [], [], []
if os.path.exists(kampala_log_path):
    log = pd.read_csv(kampala_log_path)
    if len(log) > 0:
        ids = log['image_id'].unique()[:3]
        for image_id in ids:
            rows = log[log['image_id'] == image_id]
            label = 0 if rows['label'].iloc[0] == 'real' else 1
            for _, row in rows.iterrows():
                folder = kampala_dirs.get(row['version'])
                if folder and os.path.exists(os.path.join(folder, str(row['file_name']))):
                    kampala_paths.append(os.path.join(folder, str(row['file_name'])))
                    kampala_labels.append(label)
                    kampala_meta.append({'path': '', 'label': rows['label'].iloc[0], 'source': 'kampala', 'image_id': image_id, 'version': row['version']})

all_paths = public_paths + kampala_paths
all_labels = public_labels + kampala_labels
all_meta = public_meta + kampala_meta
print(f'Total: {len(all_paths)} (public={len(public_paths)}, kampala={len(kampala_paths)})')
print(f'Labels: real={sum(1 for l in all_labels if l==0)}, ai={sum(1 for l in all_labels if l==1)}')

## 3. Tensors + test split

In [ ]:
to_tensor = transforms.ToTensor()
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

def images_to_tensor(paths, labels, img_size):
    X_list, y_list = [], []
    for p, l in zip(paths, labels):
        im = PILImage.open(p).convert('RGB')
        t = normalize(to_tensor(im))
        X_list.append(t)
        y_list.append(float(l))
        im.close()
    return torch.stack(X_list), torch.tensor(y_list, dtype=torch.float32)

X_all, y_all = images_to_tensor(all_paths, all_labels, IMG_SIZE)

is_kampala = np.array([m['source'] == 'kampala' for m in all_meta])
public_idx = np.where(~is_kampala)[0]
kampala_idx = np.where(is_kampala)[0]

from sklearn.model_selection import train_test_split
if len(public_idx) > 0:
    pub_X = X_all[public_idx]
    pub_y = y_all[public_idx]
    pub_meta = [all_meta[i] for i in public_idx]
    X_tr, X_te, y_tr, y_te, meta_tr, meta_te = train_test_split(pub_X, pub_y, pub_meta, test_size=0.2, random_state=SEED, stratify=pub_y.int().numpy())
else:
    X_tr, X_te, y_tr, y_te, meta_tr, meta_te = X_all[:0], X_all[:0], y_all[:0], y_all[:0], [], []

if len(kampala_idx) > 0:
    X_kamp = X_all[kampala_idx]
    y_kamp = y_all[kampala_idx]
    meta_kamp = [all_meta[i] for i in kampala_idx]
else:
    X_kamp, y_kamp, meta_kamp = X_all[:0], y_all[:0], []

test_ds = TensorDataset(X_te, y_te)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

print(f'test={len(test_ds)} kampala={len(X_kamp)}')
print(f'test labels: real={int((y_te==0).sum())} ai={int((y_te==1).sum())}')

## 4. Load model

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, img_size=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x.squeeze(1)

model = TinyCNN(img_size=IMG_SIZE)
model_path = os.path.join(ROOT, 'reports', 'week3_run_c_aug.pt')
if not os.path.exists(model_path):
    model_path = os.path.join(ROOT, 'reports', 'tinycnn_baseline.pt')
    print('Falling back to Week 2 baseline')

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    print(f'Loaded {model_path}')
else:
    print('WARNING: No saved model — using untrained model')

model = model.to(DEVICE)
model.eval()

## 5. Grad-CAM implementation
Hooks into the last conv layer of `features` to compute class activation maps.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.handles = []
        
        def fwd_hook(module, input, output):
            self.activations = output.detach()
        
        def bwd_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()
        
        self.handles.append(target_layer.register_forward_hook(fwd_hook))
        self.handles.append(target_layer.register_full_backward_hook(bwd_hook))
    
    def __call__(self, x, class_idx=1):
        self.model.zero_grad()
        logits = self.model(x)
        score = logits[:, class_idx] if logits.dim() > 1 else logits[class_idx]
        score.backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(x.shape[2], x.shape[3]), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam, logits.detach().cpu().numpy()
    
    def close(self):
        for h in self.handles:
            h.remove()

target_layer = model.features[-4]
gradcam = GradCAM(model, target_layer)

## 6. Faithfulness test: heat map vs random region
For each test sample: get Grad-CAM, hide top-k highlighted pixels, hide k random pixels, compare confidence drops.

In [ ]:
def hide_pixels(img_tensor, mask, fill_value=0.5):
    masked = img_tensor.clone()
    masked[mask.bool()] = fill_value
    return masked

def get_topk_mask(cam, k_fraction=0.2):
    flat = cam.flatten()
    k = max(1, int(len(flat) * k_fraction))
    threshold = np.sort(flat)[-k]
    return (cam >= threshold).astype(np.float32)

def get_random_mask(cam, k_fraction=0.2):
    flat = cam.flatten()
    k = max(1, int(len(flat) * k_fraction))
    idx = np.random.choice(len(flat), size=k, replace=False)
    mask = np.zeros_like(flat)
    mask[idx] = 1.0
    return mask.reshape(cam.shape)

test_subset = []
test_labels = []
for i in range(min(N_SAMPLES, len(test_ds))):
    x, y = test_ds[i]
    test_subset.append(x)
    test_labels.append(int(y.item()))
test_subset = torch.stack(test_subset).to(DEVICE)
test_labels = np.array(test_labels)

results = []
for i in range(len(test_subset)):
    x_single = test_subset[i:i+1]
    true_lab = test_labels[i]
    
    cam, logits = gradcam(x_single)
    prob_ai = float(1 / (1 + np.exp(-logits[0][0]))) if logits.ndim > 1 else float(1 / (1 + np.exp(-logits[0])))
    
    topk_mask = get_topk_mask(cam, 0.2)
    rand_mask = get_random_mask(cam, 0.2)
    
    topk_t = torch.tensor(topk_mask).unsqueeze(0).unsqueeze(0).to(DEVICE)
    rand_t = torch.tensor(rand_mask).unsqueeze(0).unsqueeze(0).to(DEVICE)
    
    x_topk = hide_pixels(x_single, topk_t.expand_as(x_single), fill_value=0.5)
    with torch.no_grad():
        p_topk = torch.sigmoid(model(x_topk)).cpu().numpy()[0][0]
    
    x_rand = hide_pixels(x_single, rand_t.expand_as(x_single), fill_value=0.5)
    with torch.no_grad():
        p_rand = torch.sigmoid(model(x_rand)).cpu().numpy()[0][0]
    
    drop_topk = prob_ai - p_topk
    drop_rand = prob_ai - p_rand
    
    results.append({
        'sample': i,
        'true_label': 'real' if true_lab == 0 else 'ai',
        'p_ai_original': float(prob_ai),
        'p_ai_topk_hidden': float(p_topk),
        'p_ai_random_hidden': float(p_rand),
        'drop_topk': float(drop_topk),
        'drop_random': float(drop_rand),
        'faithful': bool(drop_topk > drop_rand),
    })
    print(f'Sample {i} ({results[-1]["true_label"]}): P(AI)={prob_ai:.3f} | drop_topk={drop_topk:.3f} | drop_random={drop_rand:.3f} | faithful={drop_topk > drop_rand}')

gradcam.close()
gc.collect()

## 7. Visualize Grad-CAM + save results

In [ ]:
def denormalize(t):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return torch.clamp(t * std + mean, 0, 1)

gradcam_viz = GradCAM(model, model.features[-4])

n_rows = min(N_SAMPLES, len(test_subset))
fig, axes = plt.subplots(n_rows, 3, figsize=(9, 3 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)

for i in range(n_rows):
    x_single = test_subset[i:i+1]
    cam, logits = gradcam_viz(x_single)
    prob_ai = float(1 / (1 + np.exp(-logits[0][0]))) if logits.ndim > 1 else float(1 / (1 + np.exp(-logits[0])))
    true_lab = test_labels[i]
    pred_lab = 1 if prob_ai >= 0.5 else 0
    correct = '✓' if true_lab == pred_lab else '✗'
    
    img_np = denormalize(x_single[0]).cpu().numpy().transpose(1, 2, 0)
    
    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title(f'{true_lab}→{pred_lab} {correct}\nP(AI)={prob_ai:.2f}', fontsize=8)
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(cam, cmap='jet')
    axes[i, 1].set_title('Grad-CAM', fontsize=8)
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(img_np)
    axes[i, 2].imshow(cam, cmap='jet', alpha=0.5)
    axes[i, 2].set_title('Overlay', fontsize=8)
    axes[i, 2].axis('off')

plt.suptitle('Grad-CAM on test samples (illustrative, not proof)', fontsize=11)
plt.tight_layout()
viz_path = os.path.join(ROOT, 'reports', 'week5_gradcam_visualization.png')
plt.savefig(viz_path, dpi=150, bbox_inches='tight')
plt.close(fig)
gradcam_viz.close()

faithful_count = sum(1 for r in results if r['faithful'])
output = {
    'week': 5,
    'member': MEMBER,
    'model': model_path,
    'n_samples': len(results),
    'faithful_count': faithful_count,
    'faithful_rate': faithful_count / len(results) if results else 0.0,
    'results': results,
    'timestamp': datetime.utcnow().isoformat() + 'Z',
}
out_path = os.path.join(ROOT, 'reports', 'week5_gradcam_results.json')
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f'\n=== WEEK 5 SUMMARY ===')
print(f'Grad-CAM visualization saved to {viz_path}')
print(f'Faithfulness results saved to {out_path}')
print(f'Faithful samples: {faithful_count}/{len(results)} ({output["faithful_rate"]:.0%})')
print('A faithful heat map causes larger confidence drop when hiding highlighted vs random areas.')
gc.collect()